In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

In [19]:
np.random.seed(42)  # For reproducibility
num_leads = 1000

lead_df = pd.DataFrame({
    'Lead_ID':range(num_leads),
    'conversion_prob':np.random.uniform(0.01,0.40,num_leads),
    'potential_ARR':np.random.uniform(5000, 50000, num_leads),
    'Time_Cost': np.random.uniform(0.5, 4.0, num_leads),         # Hours to work
    'segment':np.random.choice(['Enterprise', 'SMB'], num_leads, p=[0.2,0.8])
})

In [20]:
leads_df['Expected_ARR'] = leads_df['Conversion_Prob'] * leads_df['Potential_ARR']


In [21]:
leads_df

,Lead_ID,Conversion_Prob,Potential_ARR,Time_Cost,Segment,Expected_ARR
0,0,0.156071,13330.981798,1.415970,SMB,2080.574946
1,1,0.380779,29385.542632,1.364426,SMB,11189.385181
2,2,0.295478,44282.562614,3.671891,SMB,13084.506975
3,3,0.243477,37950.119888,1.373412,SMB,9239.974085
4,4,0.070847,41295.251654,1.451824,SMB,2925.655834
...,...,...,...,...,...,...
995,995,0.045717,34562.982032,3.528535,SMB,1580.116147
996,996,0.367752,48047.657949,1.050456,SMB,17669.636453
997,997,0.063359,8103.110736,1.584258,SMB,513.407149
998,998,0.380593,7567.462452,1.515159,SMB,2880.119968


In [ ]:
### Initialize the gurobi env and model 
env = gp.Env(empty = True) #Passing empty=True creates an unstarted environment object, which defers Gurobi's initialization until you explicitly call env.start()
env.setParam('OutputFlag',0)
env.start()
model = gp.Model('creating_lead_routing_model', env = env)


* env.setParam('OutputFlag', 0): Silences Gurobi. It disables console logging while the solver runs.

* env.setParam('OutputFlag', 1) (default): Enables Gurobi logging. It displays iteration logs, gap percentages, and execution time in the terminal.

In [27]:
## create the decision variables for the leads
X = model.addVars(num_leads, vtype = GRB.BINARY,name = 'assign_leads')

This line creates a collection of binary decision variables (stored as a Gurobi tupledict), generating one 0/1 variable for every lead in num_leads.

In [31]:
X

{0: <gurobi.Var *Awaiting Model Update*>,
 1: <gurobi.Var *Awaiting Model Update*>,
 2: <gurobi.Var *Awaiting Model Update*>,
 3: <gurobi.Var *Awaiting Model Update*>,
 4: <gurobi.Var *Awaiting Model Update*>,
 5: <gurobi.Var *Awaiting Model Update*>,
 6: <gurobi.Var *Awaiting Model Update*>,
 7: <gurobi.Var *Awaiting Model Update*>,
 8: <gurobi.Var *Awaiting Model Update*>,
 9: <gurobi.Var *Awaiting Model Update*>,
 10: <gurobi.Var *Awaiting Model Update*>,
 11: <gurobi.Var *Awaiting Model Update*>,
 12: <gurobi.Var *Awaiting Model Update*>,
 13: <gurobi.Var *Awaiting Model Update*>,
 14: <gurobi.Var *Awaiting Model Update*>,
 15: <gurobi.Var *Awaiting Model Update*>,
 16: <gurobi.Var *Awaiting Model Update*>,
 17: <gurobi.Var *Awaiting Model Update*>,
 18: <gurobi.Var *Awaiting Model Update*>,
 19: <gurobi.Var *Awaiting Model Update*>,
 20: <gurobi.Var *Awaiting Model Update*>,
 21: <gurobi.Var *Awaiting Model Update*>,
 22: <gurobi.Var *Awaiting Model Update*>,
 23: <gurobi.Var *Awa

In [65]:


model.setObjective(
    gp.quicksum(X[i] * leads_df.loc[i,'Expected_ARR'] for i in range(num_leads)),
    GRB.MAXIMIZE
)

Code Fragment,Role & Description

* model.setObjective(...),Defines the mathematical expression the solver will evaluate and optimize.

* gp.quicksum(...),Gurobi’s high-performance replacement for Python's built-in sum(). It rapidly builds 
large linear expressions without creating heavy intermediate objects.

* X[i],"The binary decision variable (1 if lead i is assigned/selected, 0 if not)."
"leads_df.loc[i, 'Expected_ARR']",The revenue coefficient for lead i fetched from your Pandas DataFrame.

* X[i] * leads_df.loc[...],"Calculates the revenue contribution for lead i. If X[i]=1, it yields the full ARR; if X[i]=0, it yields 0."

* for i in range(num_leads),Loops through every lead index to include all leads in the sum.

* GRB.MAXIMIZE,"Sets the direction of optimization (maximize total value, as opposed to GRB.MINIMIZE for costs)."

In [66]:
Total_Rep_hours = 120
Min_Enterprise_leads = 15

In [67]:
## # Constraint A: Total time spent cannot exceed available rep capacity

model.addConstr(
    gp.quicksum(X[i]*lead_df.loc[i,'Time_Cost'] for i in range(num_leads)) <= Total_Rep_hours,
    name = 'capacity_constraint'
)

<gurobi.Constr *Awaiting Model Update*>

In [68]:
# Constraint B: Must select at least 15 Enterprise leads
# We create a list of indices where the segment is 'Enterprise'
enterprise_indices = leads_df[leads_df['Segment'] == 'Enterprise'].index.tolist()


In [69]:
model.addConstr(
    gp.quicksum(X[i] for i in enterprise_indices) >= Min_Enterprise_leads,
    name = 'enterprise_constraint'
)

<gurobi.Constr *Awaiting Model Update*>

In [70]:
model.optimize()

In [74]:
if model.status == GRB.OPTIMAL:
    # Extract decision variable outputs (1 if selected, 0 if skipped)
    leads_df['Is_Assigned'] = [int(X[i].X > 0.5) for i in range(num_leads)]

    # Filter the DataFrame to keep only the selected leads
    selected_leads_df = leads_df[leads_df['Is_Assigned'] == 1].copy()

    # Calculate hourly ROI to sort the daily roster in priority calling order
    selected_leads_df['ARR_Per_Hour'] = (
        selected_leads_df['Expected_ARR'] / selected_leads_df['Time_Cost']
    )
    selected_leads_df = selected_leads_df.sort_values(
        by='ARR_Per_Hour', ascending=False
    ).reset_index(drop=True)

    # Print overall portfolio metrics
    print(f"--- OPTIMIZATION SUMMARY ---")
    print(f"Total Selected Leads: {len(selected_leads_df)}")
    print(
        f"Total Hours Used:    "
        f" {selected_leads_df['Time_Cost'].sum():.2f} / {Total_Rep_hours} hours"
    )
    print(
        f"Total Expected ARR:  "
        f" ${selected_leads_df['Expected_ARR'].sum():,.2f}"
    )
    print(
        f"Enterprise Leads:    "
        f" {len(selected_leads_df[selected_leads_df['Segment'] == 'Enterprise'])}"
    )

    # Display the final Pandas DataFrame roster
    print("\n--- ASSIGNED LEADS ROSTER (Top 10 Priority Calls) ---")
    print(
        selected_leads_df[
            [
                'Lead_ID',
                'Segment',
                'Conversion_Prob',
                'Potential_ARR',
                'Expected_ARR',
                'Time_Cost',
                'ARR_Per_Hour',
            ]
        ].head(10)
    )

else:
    print(f"Model did not find an optimal solution. Status code: {model.status}")

--- OPTIMIZATION SUMMARY ---
Total Selected Leads: 114
Total Hours Used:     119.99 / 120 hours
Total Expected ARR:   $1,275,211.53
Enterprise Leads:     27

--- ASSIGNED LEADS ROSTER (Top 10 Priority Calls) ---
   Lead_ID     Segment  Conversion_Prob  Potential_ARR  Expected_ARR  \
0       11         SMB         0.388265   49445.060360  19197.778565   
1      241         SMB         0.382032   39946.858376  15260.983992   
2       80         SMB         0.346610   40385.525649  13998.040618   
3      611  Enterprise         0.321770   34289.481607  11033.328402   
4      182         SMB         0.370631   33467.985191  12404.256458   
5      327         SMB         0.248571   40536.432821  10076.185333   
6      796         SMB         0.373526   37234.552463  13908.078290   
7      685         SMB         0.227138   43790.651702   9946.500389   
8      558  Enterprise         0.304504   42793.758629  13030.850552   
9      478         SMB         0.388028   40901.614199  15870.971100

In [75]:
selected_leads_df

,Lead_ID,Conversion_Prob,Potential_ARR,Time_Cost,Segment,Expected_ARR,Is_Assigned,ARR_Per_Hour
0,11,0.388265,49445.060360,0.719286,SMB,19197.778565,1,26690.041103
1,241,0.382032,39946.858376,0.621346,SMB,15260.983992,1,24561.171432
2,80,0.346610,40385.525649,0.595086,SMB,13998.040618,1,23522.719780
3,611,0.321770,34289.481607,0.513370,Enterprise,11033.328402,1,21491.973080
4,182,0.370631,33467.985191,0.616163,SMB,12404.256458,1,20131.461435
...,...,...,...,...,...,...,...,...
109,905,0.358142,14111.625207,0.691220,SMB,5053.965214,1,7311.654433
110,308,0.320123,20275.258664,0.898434,SMB,6490.568843,1,7224.310431
111,464,0.385247,49973.617660,2.672312,Enterprise,19252.200970,1,7204.324675
112,619,0.301821,37879.272961,1.596718,Enterprise,11432.770622,1,7160.170229


In [47]:
# 7. Extract the Results
if model.status == GRB.OPTIMAL:
    print(f"Optimal Total Expected ARR: ${model.ObjVal:,.2f}")
    
    # Retrieve which leads were actually selected (where x[i] == 1)
    selected_leads = [i for i in range(num_leads) if X[i].X > 0.5]
    
    final_roster = leads_df.iloc[selected_leads]
    


    print(f"Total Leads Selected: {len(final_roster)}")
    print(f"Total Hours Used: {final_roster['Time_Cost'].sum():.1f} / {Total_Rep_hours} hours")
    print(f"Enterprise Leads Selected: {len(final_roster[final_roster['Segment'] == 'Enterprise'])}")
else:
    print("No optimal solution found. Check constraints.")

Optimal Total Expected ARR: $1,275,211.53
Total Leads Selected: 114
Total Hours Used: 120.0 / 120 hours
Enterprise Leads Selected: 27


In [ ]:
if model.status ==GRB.OPTIMAL:
    print() 

In [48]:
##

import gurobipy as gp
from gurobipy import GRB



<gurobipy.Env, Parameter changes: OutputFlag=0>

In [55]:
leads_df['Expected_ARR'] = leads_df['Conversion_Prob'] * leads_df['Potential_ARR']

In [60]:
## setup gurobi enveronment
env = gp.Env(empty = True) #Passing empty=True creates an unstarted environment object, which defers Gurobi's initialization until you explicitly call env.start
env.setParam('OutputFlag',0)
env.start()
model = gp.Model('Creating_lead_routing_model',env = env)


In [64]:
X = model.addVars(num_leads, vtype = GRB.BINARY,name = 'assign_leads')

In [ ]:
##